In [ ]:
print("Running version: assoc_v5_with_jobgroup (FULL DEBUG)")

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import os

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("Salary.csv")
print("\nDataset shape:", df.shape)

df = df.drop(columns=["Race"], errors="ignore")
df = df.dropna()

print("\nAfter cleaning:", df.shape)

# ==============================
# FEATURE ENGINEERING
# ==============================

# Salary
df["Salary_Category"] = pd.qcut(
    df["Salary"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Education
def education_category(e):
    if e == 0:
        return "High School"
    elif e == 1:
        return "Bachelor"
    elif e == 2:
        return "Master"
    elif e == 3:
        return "PhD"
    else:
        return "Unknown"

df["Education_Category"] = df["Education Level"].apply(education_category)

# Experience
def experience_category(exp):
    if exp <= 2:
        return "Entry"
    elif exp <= 5:
        return "Junior"
    elif exp <= 10:
        return "Mid"
    elif exp <= 18:
        return "Senior"
    else:
        return "Executive"

df["Experience_Category"] = df["Years of Experience"].apply(experience_category)

# ==============================
# JOB GROUPING (CORE PART)
# ==============================
def job_group(title):
    title = title.lower()

    if any(x in title for x in ["engineer", "developer", "data", "scientist", "analyst", "architect", "it", "network"]):
        return "Tech"

    elif any(x in title for x in ["manager", "director", "vp", "chief", "ceo"]):
        return "Management"

    elif any(x in title for x in ["marketing", "sales", "brand", "advertising"]):
        return "Marketing_Sales"

    elif any(x in title for x in ["hr", "human resources", "recruit"]):
        return "HR"

    elif any(x in title for x in ["finance", "financial", "account"]):
        return "Finance"

    elif any(x in title for x in ["designer", "ux", "graphic", "creative"]):
        return "Design"

    else:
        return "Operations"

df["Job_Group"] = df["Job Title"].apply(job_group)

print("\n=== Job Group Distribution ===")
print(df["Job_Group"].value_counts())

# ==============================
# ASSOCIATION DATA
# ==============================
assoc_df = df[[
    "Education_Category",
    "Experience_Category",
    "Country",
    "Job_Group",
    "Salary_Category"
]].astype(str)

print("\n=== Association Data ===")
print(assoc_df)

# ==============================
# ENCODING
# ==============================
encoded_df = pd.get_dummies(assoc_df)

print("\nEncoded columns:", len(encoded_df.columns))

# ==============================
# APRIORI
# ==============================
frequent_itemsets = apriori(
    encoded_df,
    min_support=0.03,
    use_colnames=True
)

print("\nItemsets:", len(frequent_itemsets))
print(frequent_itemsets)

# ==============================
# RULES
# ==============================
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

print("\nAll rules:", len(rules))
print(rules)

# ==============================
# FILTER SALARY RULES
# ==============================
def has_salary(x):
    return any("Salary_Category" in i for i in x)

rules = rules[rules["consequents"].apply(has_salary)]

print("\nFiltered rules:", len(rules))
print(rules[["antecedents", "consequents", "confidence", "lift"]])

# ==============================
# REMOVE MULTI-CONSEQUENT (VISIBLE)
# ==============================
print("\n=== Removing multi-consequent rules ===")
before = len(rules)

rules = rules[rules["consequents"].apply(lambda x: len(x) == 1)]

after = len(rules)

print("Before:", before)
print("After:", after)

# ==============================
# SORT
# ==============================
rules = rules.sort_values(by=["lift", "confidence"], ascending=False)

print("\n=== Final Rules ===")
print(rules[["antecedents", "consequents", "confidence", "lift"]])

# ==============================
# SAVE
# ==============================
# ==============================
# SAVE (FIXED FOR APP USAGE)
# ==============================
os.makedirs("outputs", exist_ok=True)

# Convert frozenset → list (IMPORTANT)
rules["antecedents"] = rules["antecedents"].apply(list)
rules["consequents"] = rules["consequents"].apply(list)

# Save clean format
rules.to_csv("outputs/association_rules_v5.csv", index=False)
# ==============================
# TEST
# ==============================
def extract_salary(consequents):
    for item in consequents:
        if "Salary_Category" in item:
            return item.replace("Salary_Category_", "")
    return None


def get_insight(education, experience, country, job_group, predicted_salary, rules):
    
    best_rule = None
    best_conf = 0

    # ==============================
    # STEP 1: EXACT MATCH (with country if valid)
    # ==============================
    if country != "Other":
        for _, row in rules.iterrows():
            ant = list(row["antecedents"])
            cons = list(row["consequents"])

            if (
                f"Education_Category_{education}" in ant and
                f"Experience_Category_{experience}" in ant and
                f"Job_Group_{job_group}" in ant and
                f"Country_{country}" in ant and
                f"Salary_Category_{predicted_salary}" in cons
            ):
                return (
                    f"{experience}-level professionals in {job_group} roles in {country} "
                    f"are commonly associated with {predicted_salary} salary ranges "
                    f"(Confidence: {round(row['confidence']*100, 2)}%)."
                )

    # ==============================
    # STEP 2: FALLBACK (ignore country)
    # ==============================
    for _, row in rules.iterrows():
        ant = list(row["antecedents"])
        cons = list(row["consequents"])

        if (
            f"Education_Category_{education}" in ant and
            f"Experience_Category_{experience}" in ant and
            f"Job_Group_{job_group}" in ant
        ):
            if row["confidence"] > best_conf:
                best_conf = row["confidence"]
                best_rule = row

    if best_rule is not None:
        salary = extract_salary(best_rule["consequents"])
        return (
            f"{experience}-level professionals in {job_group} roles "
            f"are most commonly associated with {salary} salary ranges "
            f"(Confidence: {round(best_conf*100, 2)}%)."
        )

    # ==============================
    # STEP 3: MINIMAL (experience only)
    # ==============================
    for _, row in rules.iterrows():
        ant = list(row["antecedents"])
        cons = list(row["consequents"])

        if f"Experience_Category_{experience}" in ant:
            salary = extract_salary(cons)
            return (
                f"{experience}-level professionals are generally associated with "
                f"{salary} salary ranges."
            )

    return "No strong pattern found."